# CARROT-KNN test-split accuracy (reviewer response)

How well does CARROT-KNN (KNN regressor on OpenAI `text-embedding-3-small`) predict per-model accuracy and cost on each dataset? All metrics are computed on the test split.

Reads the per-router prediction files written by `carrot/train_and_infer.py`. No re-training.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

os.makedirs('../plots', exist_ok=True)

DATASETS = ['routerbench', 'open-llm-lb-v2', 'sprout']

In [ ]:
def binarized_accuracy(Y_hat, Y_test, threshold=0.5):
    return ((Y_hat > threshold) == (Y_test > threshold)).mean()

def routing_accuracy(Y_hat, Y_test):
    picked = np.argmax(Y_hat, axis=1)
    return np.take_along_axis(Y_test, picked[:, None], axis=1).mean()

def oracle_accuracy(Y_test):
    return Y_test.max(axis=1).mean()

def single_best_model_accuracy(Y_test):
    return Y_test.mean(axis=0).max()  # best single model on the test set

In [ ]:
rows = []
for ds in DATASETS:
    preds_dir = f'../data/{ds}/preds'
    meta = np.load(f'{preds_dir}/meta.npy', allow_pickle=True).item()
    Y_test = np.asarray(meta['Y_test']).astype(float)
    Y_hat = np.asarray(np.load(f'{preds_dir}/Y_hat_carrot-knn.npy', allow_pickle=True)).astype(float)

    row = {
        'dataset': ds,
        'n_test': Y_test.shape[0],
        'n_models': Y_test.shape[1],
        'binarized_acc': binarized_accuracy(Y_hat, Y_test),
        'score_MAE': float(np.abs(Y_hat - Y_test).mean()),
        'score_R2': r2_score(Y_test.flatten(), Y_hat.flatten()),
        'routing_acc': routing_accuracy(Y_hat, Y_test),
        'best_single_model_acc': single_best_model_accuracy(Y_test),
        'oracle_acc': oracle_accuracy(Y_test),
    }

    # Cost metric depends on dataset.
    cost_mae = None
    if ds == 'routerbench':
        C_test = np.asarray(meta['C_test']).astype(float)
        C_hat = np.asarray(np.load(f'{preds_dir}/C_hat_carrot-knn.npy', allow_pickle=True)).astype(float)
        cost_mae = float(np.abs(C_hat - C_test).mean())
        row['cost_MAE_unit'] = '$ per query'
    elif ds == 'sprout':
        OT_test = np.asarray(meta['OT_test']).astype(float)
        OT_hat = np.asarray(np.load(f'{preds_dir}/OT_hat_carrot-knn.npy', allow_pickle=True)).astype(float)
        cost_mae = float(np.abs(OT_hat - OT_test).mean())
        row['cost_MAE_unit'] = 'output tokens'
    else:
        row['cost_MAE_unit'] = 'N/A'
    row['cost_MAE'] = cost_mae

    rows.append(row)

df = pd.DataFrame(rows).set_index('dataset')
df.round(4)

In [ ]:
display_cols = ['n_test', 'n_models', 'binarized_acc', 'score_MAE', 'score_R2',
                'routing_acc', 'best_single_model_acc', 'oracle_acc', 'cost_MAE', 'cost_MAE_unit']
out = df[display_cols].copy()
for col in ['binarized_acc', 'score_MAE', 'score_R2', 'routing_acc',
            'best_single_model_acc', 'oracle_acc', 'cost_MAE']:
    out[col] = out[col].apply(lambda v: 'N/A' if (v is None or (isinstance(v, float) and not np.isfinite(v))) else f'{v:.4f}')

md = out.to_markdown()
print(md)
with open('../plots/knn_accuracy_table.md', 'w') as f:
    f.write('# CARROT-KNN test-split accuracy\n\n')
    f.write('KNN regressor on OpenAI `text-embedding-3-small` embeddings. Metrics on the test split.\n\n')
    f.write(md)
    f.write('\n')
print('\nSaved to plots/knn_accuracy_table.md')

**Metric definitions (reviewer-facing):**
- **binarized_acc**: `((Y_hat > 0.5) == (Y_test > 0.5)).mean()` over (prompts × models). How often the KNN predicts the correctness *flag* of each (prompt, model) pair.
- **score_MAE / score_R²**: regression quality of the KNN on the continuous per-model accuracy scores (range [0, 1]).
- **routing_acc**: accuracy if we use the KNN to route — `argmax_m Y_hat[·, m]`, then take `Y_test[·, m*]`. Reported alongside `best_single_model_acc` (best always-route-to-one-model baseline) and `oracle_acc` (pick the actually-best model per prompt).
- **cost_MAE**: mean absolute error on the cost target. For Routerbench this is dollars per query (the direct cost head); for SPROUT it is the output-token count the KNN predicts (dollars are derived downstream). Open-LLM-v2 has no learned cost head, so cost_MAE is N/A.